    Тренеровка в создании массива сделок

In [2]:
# [ОБЯЗАТЕЛЕН] импорт необходимых функций <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
import numpy as np
import pandas as pd
import sys
import os

current_dir = os.getcwd()                                               # Определяем путь к текущему файлу (где выполняется код)
parent_dir = os.path.dirname(current_dir)                               # Переход на уровень выше (fc_to_mt5_migrations)
print(f"Рабочая директория проекта {parent_dir}")
config_path = os.path.join(parent_dir, "directory_config.txt")          # Определяем путь к файлу конфигурации

directories = {}                                                        # Читаем конфигурационный файл и создаём словарь с путями
if os.path.exists(config_path):
    with open(config_path, "r", encoding="utf-8") as file:
        for line in file:
            line = line.split("#")[0].strip()  # Убираем комментарии и пробелы
            if "=" in line:
                key, value = map(str.strip, line.split("=", 1))
                directories[key] = os.path.join(parent_dir, value.strip("'\""))     # Формируем абсолютный путь
else: print(f"❌ ERROR: Файл конфигурации '{config_path}' не найден.")

for key, path in directories.items(): print(f"📂 {key}: {path}")                    # Вывод всех загруженных директорий

directory_data_temp_files   = directories["directory_data_temp_files"]
directory_data_log_files    = directories["directory_data_log_files"]
libraries_path = os.path.join(directories["directory_libraries_path"])          # Формируем путь к libraries_py каталогу с библиотеками *.py

sys.path.append(libraries_path)                                 # sys.path — это список путей, где Python ищет модули при import module_name.
if libraries_path in sys.path: print(f"✅ Каталог {libraries_path} успешно добавлен в sys.path")
else: print(f"❌ Ошибка: {libraries_path} не найден в sys.path")

# Динамически импорт необходимых функций <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
file_imports = "dynamic_import_functions.py"                            # Библиотека для динамического импорта
file_imports_path = os.path.join(libraries_path, file_imports)

if os.path.exists(file_imports_path):
    import importlib
    importlib.invalidate_caches()                                       # Сбрасываем кэш перед импортом
    from dynamic_import_functions import import_functions, print_import_function_info
    print(f"\n ✅ Импорт [{file_imports}] успешен.")
else: print(f"\n ERROR: Файл '{file_imports}' не найден по пути {file_imports_path}, импорт не выполнен.\n")

modules_to_import = {                                   # Формируем словарь, с именами файлов и функциями в них
    "yar_sed_general_lib":
        [libraries_path,
                "pd_set_option",                        # Вывод ДФ
                #"list_print",                           # Печать списков
                #"df_to_csv",                            # Сохранение ДФ в CSV 
                #"file_name_with_time",
                #"CSVLoader",
                #"move_column",
                #"save_data_log_work_file"
                ],                           # Загрузка ДФ из CSV
                
    "mt5_api":
        [libraries_path,
                #"balance_0",
                #"balance_deal",
                "mt5manager",
                "manager_connect_with_control",
                "manager_disconnect_with_control",
                "admin_connect_with_control",
                "admin_disconnect_with_control",
                #"creating_position"
                ],
                  
    "update_swop_open_position":
        [libraries_path,
            "update_swop_open_position"]        
                    }

imported = import_functions(modules_to_import)          # Импортируем модули из словаря modules_to_import
print_import_function_info(modules_to_import, imported) # Выводим переменные ожидаемые импортированными функциями 

Рабочая директория проекта c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations
📂 directory_data_temp_files: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\working_data_files
📂 directory_data_log_files: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\log_data_files
📂 directory_libraries_path: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\libraries_py
✅ Каталог c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\libraries_py успешно добавлен в sys.path

 ✅ Импорт [dynamic_import_functions.py] успешен.
Импорт из 'yar_sed_general_lib' успешен: ['pd_set_option']
Импорт из 'mt5_api' успешен: ['mt5manager', 'manager_connect_with_control', 'manager_disconnect_with_control', 'admin_connect_with_control', 'admin_disconnect_with_control']
Импорт из 'update_swop_open_position' успешен: ['update_swop_open_position']

 Импортированные функции и их параметры:
Функция 'pd_set_option' из модуля 'yar_sed_general_lib' ожидает параметры: name_df:

In [ ]:
#deal = MT5Manager.MTConRouteDealer
#print("deal = MT5Manager.MTConRouteDealer =", deal)

login_list          = [55627]
beginning_of_period = 307718788     # Начало периода
end_of_period       = 1759407988    # Конец периода

#import MT5Admin

import MT5Manager
manager = imported["manager_connect_with_control"](manager = manager if 'manager' in locals() else None, pump_mode = "POSITIONS")#(pump_mode = 'POSITIONS')
deal = MT5Manager.MTDeal(manager) 
if manager:
    admin = imported["admin_connect_with_control"](admin = admin if 'admin' in locals() else None)

    if admin:
            # объект массива сделок создавать не нужно

        a = admin.DealRequestByLogins(login_list, beginning_of_period, end_of_period)

        df = pd.DataFrame([{attr: getattr(obj, attr) for attr in dir(obj) if not attr.startswith("__")} for obj in a])
        imported["pd_set_option"]("объект массива сдело", df, 30)

        print("type(a) = ", type(a), f"🔵 a = admin.DealRequestByLogins = {a}")
        b = list()
        print("type(b) = ", type(b), f"🔵 a = admin.DealRequestByLogins = {b}")

        if not isinstance(a, list): print("❌ Ошибка: массив не получен!", f"❌: {MT5Manager.LastError()}")
        elif len(a) == 0:           print("⚠️ Получен пустой массив, но это не ошибка.")
        else:
            attributes = dir(a)
            print("attributes a", attributes)

            for i in a:
                if hasattr(i, "Deal"):                                                          # Проверяем, есть ли такой атрибут
                    try:
                        i.Deal = 0                                                              # Присваиваем значение 0
                        print(f"✅ Успешно изменён Deal для объекта {i}")
                        b.append(i)
                    except AttributeError:
                        print(f"❌ Ошибка: атрибут 'Deal' только для чтения у объекта {i}")
                else:
                    print(f"⚠️ У объекта {i} нет атрибута 'Deal'")

            """
            for i in a:
                print(i)
                attributes = dir(i)
                print(attributes)"""

            results = admin.DealAddBatch(b)
            #del a[1:len(a)-1]
            #results = admin.DealUpdateBatch(a)
            print("results = ", results)
            print("❌ ", f"❌: {MT5Manager.LastError()}")


    else: print(f"MT5Admin Failed to connect to server: {MT5Manager.LastError()}")      # не удалось подключиться к серверу           
else: print(f"MT5Manager Failed to connect to server: {MT5Manager.LastError()}")        # не удалось подключиться к серверу

if imported["admin_disconnect_with_control"](admin): del admin
else: print("❌ ERROR: разъединение mt5admin c сервером НЕ удалось.")

if imported["manager_disconnect_with_control"](manager): del manager
else: print("❌ ERROR: разъединение mt5manager c сервером НЕ удалось.")

ПРИМЕР ВЫВОДА

<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>Action</th>
      <th>Clear</th>
      <th>Comment</th>
      <th>Commission</th>
      <th>ContractSize</th>
      <th>Deal</th>
      <th>Dealer</th>
      <th>Digits</th>
      <th>DigitsCurrency</th>
      <th>EnDealAction</th>
      <th>EnDealEntry</th>
      <th>EnDealReason</th>
      <th>EnTradeModifyFlags</th>
      <th>Entry</th>
      <th>ExpertID</th>
      <th>ExternalID</th>
      <th>Fee</th>
      <th>Flags</th>
      <th>Gateway</th>
      <th>Login</th>
      <th>MarketAsk</th>
      <th>MarketBid</th>
      <th>MarketLast</th>
      <th>ModificationFlags</th>
      <th>ObsoleteValue</th>
      <th>Order</th>
      <th>PositionID</th>
      <th>Price</th>
      <th>PriceGateway</th>
      <th>PricePosition</th>
      <th>PriceSL</th>
      <th>PriceTP</th>
      <th>Print</th>
      <th>Profit</th>
      <th>ProfitRaw</th>
      <th>RateMargin</th>
      <th>RateProfit</th>
      <th>Reason</th>
      <th>Storage</th>
      <th>Symbol</th>
      <th>TickSize</th>
      <th>TickValue</th>
      <th>Time</th>
      <th>TimeMsc</th>
      <th>Value</th>
      <th>Volume</th>
      <th>VolumeClosed</th>
      <th>VolumeClosedExt</th>
      <th>VolumeExt</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>0</td>
      <td>&lt;built-in method Clear of MT5Manager.MTDeal ob...</td>
      <td>#5189, RP=1.0, SP=41.0</td>
      <td>0.0</td>
      <td>100.0</td>
      <td>5022171</td>
      <td>0</td>
      <td>2</td>
      <td>2</td>
      <td>(((1)), ((2)), ((4)), ((8)), ((16)))</td>
      <td>(((1)), ((2)))</td>
      <td>(((1)), ((2)), ((4)), ((8)), ((16)))</td>
      <td>(((1)), ((2)), ((4)), ((8)), ((16)), ((32)), (...</td>
      <td>0</td>
      <td>0</td>
      <td></td>
      <td>0.0</td>
      <td>0</td>
      <td></td>
      <td>55627</td>
      <td>9.9900</td>
      <td>9.8900</td>
      <td>0.0</td>
      <td>256</td>
      <td>0.0</td>
      <td>0</td>
      <td>4953931</td>
      <td>11.9600</td>
      <td>0.0</td>
      <td>0.0000</td>
      <td>0.0</td>
      <td>0.0</td>
      <td>&lt;built-in method Print of MT5Manager.MTDeal ob...</td>
      <td>0.00</td>
      <td>0.0</td>
      <td>1.00000</td>
      <td>1.000000</td>
      <td>0</td>
      <td>0.0</td>
      <td>F.N</td>
      <td>0.0</td>
      <td>0.0</td>
      <td>1692284947</td>
      <td>1692284947293</td>
      <td>0.0</td>
      <td>7000</td>
      <td>0</td>
      <td>0</td>
      <td>70000000</td>
    </tr>
    <tr>
      <th>1</th>
      <td>1</td>
      <td>&lt;built-in method Clear of MT5Manager.MTDeal ob...</td>
      <td>#5189, PC=4.2</td>
      <td>0.0</td>
      <td>100.0</td>
      <td>5022172</td>
      <td>0</td>
      <td>2</td>
      <td>2</td>
      <td>(((1)), ((2)), ((4)), ((8)), ((16)))</td>
      <td>(((1)), ((2)))</td>
      <td>(((1)), ((2)), ((4)), ((8)), ((16)))</td>
      <td>(((1)), ((2)), ((4)), ((8)), ((16)), ((32)), (...</td>
      <td>1</td>
      <td>0</td>
      <td></td>
      <td>0.0</td>
      <td>0</td>
      <td></td>
      <td>55627</td>
      <td>9.9900</td>
      <td>9.8900</td>
      <td>0.0</td>
      <td>292</td>
      <td>0.0</td>
      <td>0</td>
      <td>4953931</td>
      <td>12.0200</td>
      <td>0.0</td>
      <td>11.9600</td>
      <td>0.0</td>
      <td>0.0</td>
      <td>&lt;built-in method Print of MT5Manager.MTDeal ob...</td>
      <td>4.20</td>
      <td>4.2</td>
      <td>1.00000</td>
      <td>1.000000</td>
      <td>0</td>
      <td>41.0</td>
      <td>F.N</td>
      <td>0.0</td>
      <td>0.0</td>
      <td>1692624752</td>
      <td>1692624752614</td>
      <td>0.0</td>
      <td>7000</td>
      <td>7000</td>
      <td>70000000</td>
      <td>70000000</td>
    </tr>
    <tr>
      <th>2</th>
      <td>0</td>
      <td>&lt;built-in method Clear of MT5Manager.MTDeal ob...</td>
      <td>#5190, RP=1.09, SP=50.0</td>
      <td>0.0</td>
      <td>100.0</td>
      <td>5022173</td>
      <td>0</td>
      <td>2</td>
      <td>2</td>
      <td>(((1)), ((2)), ((4)), ((8)), ((16)))</td>
      <td>(((1)), ((2)))</td>
      <td>(((1)), ((2)), ((4)), ((8)), ((16)))</td>
      <td>(((1)), ((2)), ((4)), ((8)), ((16)), ((32)), (...</td>
      <td>0</td>
      <td>0</td>
      <td></td>
      <td>0.0</td>
      <td>0</td>
      <td></td>
      <td>55627</td>
      <td>14.5200</td>
      <td>14.2500</td>
      <td>0.0</td>
      <td>256</td>
      <td>0.0</td>
      <td>0</td>
      <td>4953932</td>
      <td>10.6100</td>
      <td>0.0</td>
      <td>0.0000</td>
      <td>0.0</td>
      <td>0.0</td>
      <td>&lt;built-in method Print of MT5Manager.MTDeal ob...</td>
      <td>0.00</td>
      <td>0.0</td>
      <td>1.07978</td>
      <td>1.079080</td>
      <td>0</td>
      <td>0.0</td>
      <td>IBE.MAC</td>
      <td>0.0</td>
      <td>0.0</td>
      <td>1692285023</td>
      <td>1692285023734</td>
      <td>0.0</td>
      <td>10000</td>
      <td>0</td>
      <td>0</td>
      <td>100000000</td>
    </tr>
    <tr>
      <th>3</th>
      <td>1</td>
      <td>&lt;built-in method Clear of MT5Manager.MTDeal ob...</td>
      <td>#5190, PC=4.35</td>
      <td>0.0</td>
      <td>100.0</td>
      <td>5022174</td>
      <td>0</td>
      <td>2</td>
      <td>2</td>
      <td>(((1)), ((2)), ((4)), ((8)), ((16)))</td>
      <td>(((1)), ((2)))</td>
      <td>(((1)), ((2)), ((4)), ((8)), ((16)))</td>
      <td>(((1)), ((2)), ((4)), ((8)), ((16)), ((32)), (...</td>
      <td>1</td>
      <td>0</td>
      <td></td>
      <td>0.0</td>
      <td>0</td>
      <td></td>
      <td>55627</td>
      <td>14.5200</td>
      <td>14.2500</td>
      <td>0.0</td>
      <td>292</td>
      <td>0.0</td>
      <td>0</td>
      <td>4953932</td>
      <td>10.6500</td>
      <td>0.0</td>
      <td>10.6100</td>
      <td>0.0</td>
      <td>0.0</td>
      <td>&lt;built-in method Print of MT5Manager.MTDeal ob...</td>
      <td>4.35</td>
      <td>4.0</td>
      <td>1.07910</td>
      <td>1.087500</td>
      <td>0</td>
      <td>50.0</td>
      <td>IBE.MAC</td>
      <td>0.0</td>
      <td>0.0</td>
      <td>1692624752</td>
      <td>1692624752224</td>
      <td>0.0</td>
      <td>10000</td>
      <td>10000</td>
      <td>100000000</td>
      <td>100000000</td>
    </tr>
    <tr>
      <th>4</th>
      <td>2</td>
      <td>&lt;built-in method Clear of MT5Manager.MTDeal ob...</td>
      <td>#46793, balance, deposit</td>
      <td>0.0</td>
      <td>0.0</td>
      <td>5022196</td>
      <td>1100</td>
      <td>2</td>
      <td>2</td>
      <td>(((1)), ((2)), ((4)), ((8)), ((16)))</td>
      <td>(((1)), ((2)))</td>
      <td>(((1)), ((2)), ((4)), ((8)), ((16)))</td>
      <td>(((1)), ((2)), ((4)), ((8)), ((16)), ((32)), (...</td>
      <td>0</td>
      <td>0</td>
      <td></td>
      <td>0.0</td>
      <td>0</td>
      <td></td>
      <td>55627</td>
      <td>0.0000</td>
      <td>0.0000</td>
      <td>0.0</td>
      <td>32</td>
      <td>0.0</td>
      <td>0</td>
      <td>0</td>
      <td>0.0000</td>
      <td>0.0</td>
      <td>0.0000</td>
      <td>0.0</td>
      <td>0.0</td>
      <td>&lt;built-in method Print of MT5Manager.MTDeal ob...</td>
      <td>250.00</td>
      <td>0.0</td>
      <td>0.00000</td>
      <td>0.000000</td>
      <td>2</td>
      <td>0.0</td>
      <td></td>
      <td>0.0</td>
      <td>0.0</td>
      <td>1692295511</td>
      <td>1692295511000</td>
      <td>0.0</td>
      <td>0</td>
      <td>0</td>
      <td>0</td>
      <td>0</td>
    </tr>
    <tr>
      <th>...</th>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
    </tr>
    <tr>
      <th>711</th>
      <td>0</td>
      <td>&lt;built-in method Clear of MT5Manager.MTDeal ob...</td>
      <td>#179324, PC=32.1</td>
      <td>0.0</td>
      <td>100.0</td>
      <td>5336635</td>
      <td>0</td>
      <td>3</td>
      <td>2</td>
      <td>(((1)), ((2)), ((4)), ((8)), ((16)))</td>
      <td>(((1)), ((2)))</td>
      <td>(((1)), ((2)), ((4)), ((8)), ((16)))</td>
      <td>(((1)), ((2)), ((4)), ((8)), ((16)), ((32)), (...</td>
      <td>1</td>
      <td>0</td>
      <td></td>
      <td>0.0</td>
      <td>0</td>
      <td></td>
      <td>55627</td>
      <td>3024.0250</td>
      <td>3023.9770</td>
      <td>0.0</td>
      <td>0</td>
      <td>0.0</td>
      <td>0</td>
      <td>5061943</td>
      <td>2663.5700</td>
      <td>0.0</td>
      <td>2666.7800</td>
      <td>0.0</td>
      <td>0.0</td>
      <td>&lt;built-in method Print of MT5Manager.MTDeal ob...</td>
      <td>32.10</td>
      <td>32.1</td>
      <td>1.00000</td>
      <td>1.000000</td>
      <td>0</td>
      <td>-45.0</td>
      <td>XAUUSD</td>
      <td>0.0</td>
      <td>0.0</td>
      <td>1732209588</td>
      <td>1732209588160</td>
      <td>0.0</td>
      <td>1000</td>
      <td>1000</td>
      <td>10000000</td>
      <td>10000000</td>
    </tr>
    <tr>
      <th>712</th>
      <td>1</td>
      <td>&lt;built-in method Clear of MT5Manager.MTDeal ob...</td>
      <td>#179326, RP=1.0, SP=-270.0</td>
      <td>0.0</td>
      <td>10.0</td>
      <td>5336636</td>
      <td>0</td>
      <td>2</td>
      <td>2</td>
      <td>(((1)), ((2)), ((4)), ((8)), ((16)))</td>
      <td>(((1)), ((2)))</td>
      <td>(((1)), ((2)), ((4)), ((8)), ((16)))</td>
      <td>(((1)), ((2)), ((4)), ((8)), ((16)), ((32)), (...</td>
      <td>0</td>
      <td>0</td>
      <td></td>
      <td>0.0</td>
      <td>0</td>
      <td></td>
      <td>55627</td>
      <td>1985.5300</td>
      <td>1980.4400</td>
      <td>0.0</td>
      <td>0</td>
      <td>0.0</td>
      <td>0</td>
      <td>5061945</td>
      <td>3277.8630</td>
      <td>0.0</td>
      <td>0.0000</td>
      <td>0.0</td>
      <td>0.0</td>
      <td>&lt;built-in method Print of MT5Manager.MTDeal ob...</td>
      <td>0.00</td>
      <td>0.0</td>
      <td>3277.86300</td>
      <td>1.000000</td>
      <td>0</td>
      <td>0.0</td>
      <td>ETHUSD</td>
      <td>0.0</td>
      <td>0.0</td>
      <td>1732204789</td>
      <td>1732204789286</td>
      <td>0.0</td>
      <td>3000</td>
      <td>0</td>
      <td>0</td>
      <td>30000000</td>
    </tr>
    <tr>
      <th>713</th>
      <td>0</td>
      <td>&lt;built-in method Clear of MT5Manager.MTDeal ob...</td>
      <td>#179326, PC=-115.0</td>
      <td>0.0</td>
      <td>10.0</td>
      <td>5336637</td>
      <td>0</td>
      <td>2</td>
      <td>2</td>
      <td>(((1)), ((2)), ((4)), ((8)), ((16)))</td>
      <td>(((1)), ((2)))</td>
      <td>(((1)), ((2)), ((4)), ((8)), ((16)))</td>
      <td>(((1)), ((2)), ((4)), ((8)), ((16)), ((32)), (...</td>
      <td>1</td>
      <td>0</td>
      <td></td>
      <td>0.0</td>
      <td>0</td>
      <td></td>
      <td>55627</td>
      <td>1985.7500</td>
      <td>1980.6800</td>
      <td>0.0</td>
      <td>0</td>
      <td>0.0</td>
      <td>0</td>
      <td>5061945</td>
      <td>3316.1979</td>
      <td>0.0</td>
      <td>3277.8630</td>
      <td>0.0</td>
      <td>0.0</td>
      <td>&lt;built-in method Print of MT5Manager.MTDeal ob...</td>
      <td>-115.00</td>
      <td>-115.0</td>
      <td>3316.19790</td>
      <td>0.999959</td>
      <td>0</td>
      <td>-270.0</td>
      <td>ETHUSD</td>
      <td>0.0</td>
      <td>0.0</td>
      <td>1732209269</td>
      <td>1732209269034</td>
      <td>0.0</td>
      <td>3000</td>
      <td>3000</td>
      <td>30000000</td>
      <td>30000000</td>
    </tr>
    <tr>
      <th>714</th>
      <td>1</td>
      <td>&lt;built-in method Clear of MT5Manager.MTDeal ob...</td>
      <td>#179329, RP=1.0, SP=-991.0</td>
      <td>0.0</td>
      <td>10000.0</td>
      <td>5336638</td>
      <td>0</td>
      <td>4</td>
      <td>2</td>
      <td>(((1)), ((2)), ((4)), ((8)), ((16)))</td>
      <td>(((1)), ((2)))</td>
      <td>(((1)), ((2)), ((4)), ((8)), ((16)))</td>
      <td>(((1)), ((2)), ((4)), ((8)), ((16)), ((32)), (...</td>
      <td>0</td>
      <td>0</td>
      <td></td>
      <td>0.0</td>
      <td>0</td>
      <td></td>
      <td>55627</td>
      <td>4.4448</td>
      <td>4.4319</td>
      <td>0.0</td>
      <td>0</td>
      <td>0.0</td>
      <td>0</td>
      <td>5061948</td>
      <td>8.7713</td>
      <td>0.0</td>
      <td>0.0000</td>
      <td>0.0</td>
      <td>0.0</td>
      <td>&lt;built-in method Print of MT5Manager.MTDeal ob...</td>
      <td>0.00</td>
      <td>0.0</td>
      <td>1.00000</td>
      <td>1.000000</td>
      <td>0</td>
      <td>0.0</td>
      <td>Bitwise_LTD</td>
      <td>0.0</td>
      <td>0.0</td>
      <td>1732204876</td>
      <td>1732204876468</td>
      <td>0.0</td>
      <td>2000</td>
      <td>0</td>
      <td>0</td>
      <td>20000000</td>
    </tr>
    <tr>
      <th>715</th>
      <td>0</td>
      <td>&lt;built-in method Clear of MT5Manager.MTDeal ob...</td>
      <td>#179329, PC=-1017.2</td>
      <td>0.0</td>
      <td>10000.0</td>
      <td>5336639</td>
      <td>0</td>
      <td>4</td>
      <td>2</td>
      <td>(((1)), ((2)), ((4)), ((8)), ((16)))</td>
      <td>(((1)), ((2)))</td>
      <td>(((1)), ((2)), ((4)), ((8)), ((16)))</td>
      <td>(((1)), ((2)), ((4)), ((8)), ((16)), ((32)), (...</td>
      <td>1</td>
      <td>0</td>
      <td></td>
      <td>0.0</td>
      <td>0</td>
      <td></td>
      <td>55627</td>
      <td>4.4448</td>
      <td>4.4319</td>
      <td>0.0</td>
      <td>0</td>
      <td>0.0</td>
      <td>0</td>
      <td>5061948</td>
      <td>9.2799</td>
      <td>0.0</td>
      <td>8.7713</td>
      <td>0.0</td>
      <td>0.0</td>
      <td>&lt;built-in method Print of MT5Manager.MTDeal ob...</td>
      <td>-1017.20</td>
      <td>-1017.2</td>
      <td>1.00000</td>
      <td>1.000000</td>
      <td>0</td>
      <td>-991.0</td>
      <td>Bitwise_LTD</td>
      <td>0.0</td>
      <td>0.0</td>
      <td>1732209231</td>
      <td>1732209231902</td>
      <td>0.0</td>
      <td>2000</td>
      <td>2000</td>
      <td>20000000</td>
      <td>20000000</td>
    </tr>
  </tbody>
</table>
<p>716 rows × 49 columns</p>
</div>